# SIMURGH · GMD-26 trajectory force-error gallery

Interactive per-frame force-error playback for a **trained SIMURGH (ARACE) model** on the
GMD-26 *Fragment Duplication* benchmark — the goal.ml port of the MACE-based
`gmd_task_video_gallery_py3dmol.ipynb`, which pulled its calculators from the external
SCAI/GMD project. Everything here comes from this repository: `goal.ml` for the model,
`data/GMD/` for the trajectories, `logs/` for the checkpoint.

**Encoding** (same as the original, so the two are visually comparable):

- **inner sphere** — relative *magnitude* error `|‖F_pred‖ − ‖F_ref‖| / ‖F_ref‖`
  (white → pink), scaled to that trajectory's 1st–99th percentile
- **outer shell** — *directional* error via cosine similarity `cos(F_pred, F_ref)`
  (white = aligned → red → yellow = opposite), opacity ∝ how wrong the direction is
- every frame is rigidly (Kabsch) aligned to frame 0; oxygens (`O1`, `O2`, …) and the
  first carbon (`C1`) are labelled

**Scope** — 5 trajectories, one viewer each: 2 in-distribution (monocarboxylic acids, the
family the model was trained on) and 3 out-of-distribution (dicarboxylic acids — the
duplicated functional group the benchmark tests). Section 6 aggregates all five into one
summary table; section 7 shows the same model as a plain ASE calculator.

In [ ]:
from __future__ import annotations

import io
import json
import time
from pathlib import Path

import ase
import ase.io
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import torch
from ase.neighborlist import NeighborList, natural_cutoffs
from matplotlib.cm import ScalarMappable
from torch_geometric.data import Batch

import ipywidgets as widgets
import py3Dmol
from IPython.display import HTML, display

# Populates the GOAL registries and patches Lightning's checkpoint IO
# (GOAL checkpoints carry an OmegaConf config, so weights_only=False is required).
import goal.ml.cli  # noqa: F401
from goal.ml.data.graph import AtomicGraph
from goal.ml.utils.calculator import GOALCalculator

print(f"torch {torch.__version__} · CUDA available: {torch.cuda.is_available()}")

## 0 · Configuration

The checkpoint directory is written out explicitly rather than looked up by task name —
same principle as the original notebook: *which* weights produced the pictures below
should be readable off the page, not hidden behind a resolver. Point `CKPT_DIR` at any
other run under `logs/` to re-render the whole gallery for a different model.

In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "GMD" / "FragmentDuplication"

# Trained SIMURGH-ARACE run (best val_forces_mae ≈ 0.0114 eV/Å, epoch ~776).
CKPT_DIR = (
    PROJECT_ROOT
    / "logs"
    / "simurgh_gmd26"
    / "runs"
    / "2026-08-03_16-52-12_trajectory_simurgh_arace_fragment_duplication"
    / "checkpoints"
)

# The five trajectories the gallery renders: (split, file).
TRAJECTORIES: list[tuple[str, str]] = [
    ("TestID", "O2C4H8_PBE.traj"),  # ID  — shortest monocarboxylic acid
    ("TestID", "O2C10H20_PBE.traj"),  # ID  — longest monocarboxylic acid
    ("TestOOD", "O4C4H6_PBE.traj"),  # OOD — shortest dicarboxylic acid
    ("TestOOD", "O4C7H12_PBE.traj"),  # OOD — mid-chain dicarboxylic acid
    ("TestOOD", "O4C10H18_PBE.traj"),  # OOD — longest dicarboxylic acid
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FRAME_STRIDE = 1  # >1 subsamples frames (each trajectory holds 200)
BATCH_SIZE = 4  # frames per forward pass; auto-halves on CUDA OOM
PLAYBACK_FPS = 3

assert CKPT_DIR.is_dir(), f"checkpoint dir not found: {CKPT_DIR}"
assert DATA_ROOT.is_dir(), f"GMD data not found: {DATA_ROOT}"
print(f"checkpoints : {CKPT_DIR}")
print(f"data        : {DATA_ROOT}")
print(f"device      : {DEVICE}")

## 1 · Load the trained SIMURGH model

`GOALCalculator` accepts the *checkpoint directory* (format B) and loads through the
`frozen_source/` snapshot saved alongside the weights, so the model is rebuilt from the
code as it was at training time — immune to later edits in `src/goal/`.
`checkpoint="best"` reads `checkpoint_state.json` and picks the top-k entry with the
lowest monitored metric.

We keep the calculator (section 7 uses it as an ASE calculator) but do the bulk work
straight against `calc._module`, which lets us batch frames instead of paying ASE's
one-structure-at-a-time overhead.

In [ ]:
t0 = time.perf_counter()
calc = GOALCalculator(checkpoint_path=CKPT_DIR, checkpoint="best", device=DEVICE)
module = calc._module
CUTOFF = calc._cutoff
DTYPE = calc.dtype  # sniffed from the model's own parameters
print(f"loaded in {time.perf_counter() - t0:.1f}s\n")

metadata = json.loads((CKPT_DIR / "metadata.json").read_text())
config = torch.load(
    str(CKPT_DIR / "last.ckpt"), map_location="cpu", weights_only=False
)["hyper_parameters"]["config"]

print(f"model class   : {metadata['model_class']}")
print(f"parameters    : {sum(p.numel() for p in module.parameters()):,}")
print(f"elements (Z)  : {metadata['elements']}")
print(f"cutoff        : {CUTOFF} Å   dtype: {DTYPE}")
print(f"backbone      : {config.model.backbone.name}, {config.model.backbone.num_rounds} rounds, "
      f"irreps {config.model.backbone.artisan.hidden_irreps}")
print(f"head          : {config.model.head.name}  (autograd forces)")
print(f"trained on    : {Path(str(config.data.train_dir)).parent.name}/"
      f"{Path(str(config.data.train_dir)).name}")

## 2 · Batched inference

One wrinkle specific to this model: its backbone enables `fragment_interaction`, so every
graph must carry `fragment_index` — the per-atom fragment labels produced by cutting the
SMARTS bonds listed in `data.fragment_*`. `AtomicGraph.from_ase` computes them on request,
but `GOALCalculator.calculate` never asks, so a stock calculator raises
*"fragment_interaction is enabled but the graphs carry no 'fragment_index'"*. Reading the
fragment settings back off the checkpoint's own config guarantees inference builds graphs
exactly the way training did.

In [ ]:
FRAGMENT_KWARGS: dict[str, object] = {
    "compute_fragment_index": bool(config.data.get("compute_fragment_index", False)),
    "fragment_covalent_cutoff": float(config.data.get("fragment_covalent_cutoff", 1.8)),
    "fragment_scheme": str(config.data.get("fragment_scheme", "connected")),
    "fragment_charge": int(config.data.get("fragment_charge", 0)),
    "fragment_on_failure": str(config.data.get("fragment_on_failure", "fallback")),
    "fragment_smarts": (
        list(config.data.fragment_smarts) if config.data.get("fragment_smarts") else None
    ),
    "fragment_keep_groups": (
        list(config.data.fragment_keep_groups)
        if config.data.get("fragment_keep_groups")
        else None
    ),
}
# The `custom` scheme runs rdkit bond perception before cutting the SMARTS bonds, and rdkit
# refuses some geometries ("unable to determine valence …") — every frame of the five
# trajectories below perceives cleanly, but a mid-relaxation or mid-MD geometry need not.
# Training could therefore afford `fragment_on_failure: raise`; inference cannot, so failures
# degrade to connected-component labels instead of aborting. Those frames are counted, since
# a different decomposition means the fragment channel saw something training never did.
FALLBACK_FRAGMENT_KWARGS = {**FRAGMENT_KWARGS, "fragment_scheme": "connected",
                            "fragment_smarts": None, "fragment_on_failure": "fallback"}
FRAGMENT_FALLBACKS = 0

print("fragment settings from the checkpoint config:")
for key, value in FRAGMENT_KWARGS.items():
    print(f"  {key:26s} {value}")


def to_graph(atoms: ase.Atoms) -> AtomicGraph:
    """ASE frame → AtomicGraph, built the same way the training dataloader built it."""
    global FRAGMENT_FALLBACKS
    try:
        return AtomicGraph.from_ase(atoms, cutoff=CUTOFF, dtype=DTYPE, **FRAGMENT_KWARGS)
    except Exception:  # noqa: BLE001 — rdkit perception failure, see the comment above
        FRAGMENT_FALLBACKS += 1
        return AtomicGraph.from_ase(atoms, cutoff=CUTOFF, dtype=DTYPE, **FALLBACK_FRAGMENT_KWARGS)


def _forward(chunk: list[ase.Atoms]) -> tuple[np.ndarray, list[np.ndarray]]:
    """Energies + per-frame forces for one batch of frames."""
    batch = Batch.from_data_list([to_graph(a) for a in chunk]).to(DEVICE)
    # enable_grad, not no_grad: EnergyForcesHead differentiates the energy w.r.t. positions.
    with torch.enable_grad():
        out = module(batch)
    energies = out["energy"].detach().cpu().numpy()
    forces = out["forces"].detach().cpu().numpy()
    del out, batch
    if DEVICE == "cuda":
        # The autograd graph is retained inside the head; drop it before the next batch,
        # otherwise peak memory is ~2 batches' worth.
        torch.cuda.empty_cache()
    split_at = np.cumsum([len(a) for a in chunk])[:-1]
    return energies, np.split(forces, split_at)


def predict_frames(
    frames: list[ase.Atoms], batch_size: int = BATCH_SIZE
) -> tuple[np.ndarray, list[np.ndarray]]:
    """Predict energies/forces for every frame, halving the batch size on CUDA OOM.

    A ~30-atom GMD frame costs roughly 0.5 GB of activations, so the back-off matters when
    the GPU is shared with a training run.
    """
    energies: list[np.ndarray] = []
    forces: list[np.ndarray] = []
    start, bs = 0, batch_size
    while start < len(frames):
        try:
            e, f = _forward(frames[start : start + bs])
        except torch.OutOfMemoryError:
            if bs == 1:
                raise
            bs = max(1, bs // 2)
            torch.cuda.empty_cache()
            print(f"  CUDA OOM — retrying with batch_size={bs}")
            continue
        energies.append(e)
        forces.extend(f)
        start += bs
    return np.concatenate(energies), forces


# Every trajectory is predicted at most once; the viewer and the summary share this cache.
PREDICTIONS: dict[tuple[str, str], dict[str, object]] = {}


def evaluate_trajectory(split: str, molecule: str, stride: int = FRAME_STRIDE) -> dict[str, object]:
    """Load one trajectory, predict every frame, and derive the per-atom error fields."""
    key = (split, molecule)
    if key in PREDICTIONS:
        return PREDICTIONS[key]

    path = DATA_ROOT / split / molecule
    if not path.is_file():
        available = sorted(p.name for p in (DATA_ROOT / split).glob("*.traj"))
        raise FileNotFoundError(f"{path} not found. Available: {available}")

    frames = ase.io.read(str(path), index=f"::{stride}")
    ref_energy = np.array([f.get_potential_energy() for f in frames])
    ref_forces = [f.get_forces().copy() for f in frames]

    fallbacks_before = FRAGMENT_FALLBACKS
    t_start = time.perf_counter()
    pred_energy, pred_forces = predict_frames(frames)
    elapsed = time.perf_counter() - t_start
    fallbacks = FRAGMENT_FALLBACKS - fallbacks_before

    n_frames, n_atoms = len(frames), len(frames[0])
    rel_mag = np.zeros((n_frames, n_atoms))
    cos_sim = np.zeros((n_frames, n_atoms))
    for i, (pred, ref) in enumerate(zip(pred_forces, ref_forces)):
        norm_pred = np.linalg.norm(pred, axis=1)
        norm_ref = np.linalg.norm(ref, axis=1)
        denom = norm_pred * norm_ref
        cos_sim[i] = np.divide(
            (pred * ref).sum(axis=1), denom, out=np.zeros_like(denom), where=denom > 1e-20
        )
        rel_mag[i] = np.abs(norm_pred - norm_ref) / (norm_ref + 1e-10)

    positions = np.array([f.get_positions() for f in frames])
    record: dict[str, object] = {
        "split": split,
        "molecule": molecule,
        "formula": frames[0].get_chemical_formula(),
        "symbols": frames[0].get_chemical_symbols(),
        "n_frames": n_frames,
        "n_atoms": n_atoms,
        "positions": positions,
        "fragment_index": to_graph(frames[0]).fragment_index.tolist(),
        "ref_energy": ref_energy,
        "pred_energy": pred_energy,
        "ref_forces": ref_forces,
        "pred_forces": pred_forces,
        "rel_mag": rel_mag,
        "cos_sim": cos_sim,
        "energy_mae_per_atom": float(np.abs(pred_energy - ref_energy).mean() / n_atoms),
        "forces_mae": float(np.mean([np.abs(p - r).mean() for p, r in zip(pred_forces, ref_forces)])),
        "fragment_fallbacks": fallbacks,
        "seconds": elapsed,
    }
    PREDICTIONS[key] = record
    note = f" · {fallbacks} frame(s) fell back to connected fragments" if fallbacks else ""
    print(
        f"{split}/{molecule}: {n_frames} frames × {n_atoms} atoms in {elapsed:.1f}s "
        f"({elapsed / n_frames * 1e3:.0f} ms/frame){note}"
    )
    return record

## 3 · Sanity check — one frame against DFT

Before rendering anything, confirm the loaded model reproduces its training-time accuracy
on a single in-distribution frame. `val_forces_mae` for this checkpoint was ≈ 0.0114 eV/Å.

In [ ]:
probe = ase.io.read(str(DATA_ROOT / "TestID" / "O2C6H12_PBE.traj"), index=100)
probe_ref_e = probe.get_potential_energy()
probe_ref_f = probe.get_forces().copy()
probe_pred_e, (probe_pred_f,) = predict_frames([probe])

norm_pred = np.linalg.norm(probe_pred_f, axis=1)
norm_ref = np.linalg.norm(probe_ref_f, axis=1)
probe_cos = (probe_pred_f * probe_ref_f).sum(axis=1) / (norm_pred * norm_ref + 1e-12)

print(f"{probe.get_chemical_formula()} · frame 100 · {len(probe)} atoms\n")
print(f"{'':22s} {'DFT':>14s} {'SIMURGH':>14s}")
print(f"{'energy (eV)':22s} {probe_ref_e:14.4f} {probe_pred_e[0]:14.4f}")
print(f"{'|energy| / atom (eV)':22s} {'':14s} {abs(probe_pred_e[0] - probe_ref_e) / len(probe):14.5f}")
print(f"{'forces MAE (eV/Å)':22s} {'':14s} {np.abs(probe_pred_f - probe_ref_f).mean():14.5f}")
print(f"{'mean cos(F_pred,F_ref)':22s} {'':14s} {probe_cos.mean():14.5f}")
print(f"{'net force ‖ΣF‖ (eV/Å)':22s} {np.linalg.norm(probe_ref_f.sum(axis=0)):14.2e} "
      f"{np.linalg.norm(probe_pred_f.sum(axis=0)):14.2e}")

## 4 · Viewer

Straight port of the original helper: Kabsch alignment, the two colour maps, and a
`Play ▶ + slider` pair driving one py3Dmol view per frame. The only change is that the
forces now come from `evaluate_trajectory` (SIMURGH) instead of a MACE calculator.

In [ ]:
CMAP_MAG = mcolors.LinearSegmentedColormap.from_list("mag_err", ["#ffffff", "#ffb6d5", "#ff1493"])
# Point colour parameterised by t = (1 − cos)/2 → t=0 (cos=+1) white, t=0.5 red, t=1 yellow.
CMAP_DIR = mcolors.LinearSegmentedColormap.from_list(
    "dir_err", [(0.0, "#ffffff"), (0.5, "#e60026"), (1.0, "#ffd000")]
)
# Colorbar version, plotted against cos ∈ [−1, +1] → cos=−1 yellow … cos=+1 white.
CMAP_DIR_CB = mcolors.LinearSegmentedColormap.from_list(
    "dir_err_cb", [(0.0, "#ffd000"), (0.5, "#e60026"), (1.0, "#ffffff")]
)
OUTER_OPACITY_MIN, OUTER_OPACITY_MAX = 0.25, 0.85
VIEWER_WIDTH, VIEWER_HEIGHT = 600, 460


def kabsch_align(mobile: np.ndarray, ref: np.ndarray) -> np.ndarray:
    """Rigidly rotate + translate `mobile` (n_atoms, 3) onto `ref`, minimising RMSD."""
    mobile_com = mobile.mean(axis=0)
    ref_com = ref.mean(axis=0)
    H = (mobile - mobile_com).T @ (ref - ref_com)
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    return (R @ (mobile - mobile_com).T).T + ref_com


def colorbar_figure(norm, cmap, label):
    fig, ax = plt.subplots(figsize=(1.0, 3.6))
    fig.subplots_adjust(left=0.45, right=0.6, top=0.92, bottom=0.1)
    fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=ax, orientation="vertical").set_label(
        label, fontsize=8
    )
    plt.close(fig)
    return fig


def show_trajectory(split: str, molecule: str) -> None:
    """Render the interactive force-error playback for one GMD trajectory."""
    record = evaluate_trajectory(split, molecule)
    symbols: list[str] = record["symbols"]  # type: ignore[assignment]
    n_atoms: int = record["n_atoms"]  # type: ignore[assignment]
    n_frames: int = record["n_frames"]  # type: ignore[assignment]
    rel_mag: np.ndarray = record["rel_mag"]  # type: ignore[assignment]
    cos_sim: np.ndarray = record["cos_sim"]  # type: ignore[assignment]

    # Align every frame onto frame 0 so the molecule stops tumbling during playback.
    positions = np.array(record["positions"])
    reference = positions[0].copy()
    positions = np.array([kabsch_align(p, reference) for p in positions])

    mag_vmin, mag_vmax = np.percentile(rel_mag, [1, 99])
    if mag_vmax <= mag_vmin:
        mag_vmax = mag_vmin + 1e-6

    def two_sphere_colors(frame_mag: np.ndarray, frame_cos: np.ndarray):
        inner = [mcolors.to_hex(CMAP_MAG(min(float(m) / mag_vmax, 1.0))) for m in frame_mag]
        t_dir = (1.0 - frame_cos) / 2.0
        outer = [mcolors.to_hex(CMAP_DIR(t)) for t in t_dir]
        opacity = OUTER_OPACITY_MIN + t_dir * (OUTER_OPACITY_MAX - OUTER_OPACITY_MIN)
        return inner, outer, opacity

    # Bonds are taken from frame 0 and reused — covalent connectivity does not change.
    first_frame = ase.Atoms(symbols=symbols, positions=positions[0])
    nl = NeighborList(natural_cutoffs(first_frame, mult=1.1), self_interaction=False, bothways=False)
    nl.update(first_frame)
    bonds = [(i, j) for i in range(n_atoms) for j in nl.get_neighbors(i)[0] if j > i]

    oxygen_idx = [i for i, s in enumerate(symbols) if s == "O"]
    first_carbon_idx = next((i for i, s in enumerate(symbols) if s == "C"), None)
    tracked_labels = {idx: f"O{n + 1}" for n, idx in enumerate(oxygen_idx)}
    if first_carbon_idx is not None:
        tracked_labels[first_carbon_idx] = "C1"

    def build_view(frame_idx: int) -> py3Dmol.view:
        frame_pos = positions[frame_idx]
        inner, outer, opacity = two_sphere_colors(rel_mag[frame_idx], cos_sim[frame_idx])

        buf = io.StringIO()
        ase.io.write(buf, ase.Atoms(symbols=symbols, positions=frame_pos), format="xyz")

        view = py3Dmol.view(width=VIEWER_WIDTH, height=VIEWER_HEIGHT)
        view.addModel(buf.getvalue(), "xyz")
        view.setStyle({}, {})  # suppress py3Dmol's own bonds; we draw them below

        for i, j in bonds:
            p1, p2 = frame_pos[i], frame_pos[j]
            view.addCylinder({
                "start": {"x": float(p1[0]), "y": float(p1[1]), "z": float(p1[2])},
                "end": {"x": float(p2[0]), "y": float(p2[1]), "z": float(p2[2])},
                "radius": 0.08,
                "color": "#888888",
                "fromCap": True,
                "toCap": True,
            })

        for i, pos in enumerate(frame_pos):
            x, y, z = float(pos[0]), float(pos[1]), float(pos[2])
            view.addSphere({"center": {"x": x, "y": y, "z": z}, "radius": 0.42,
                            "color": outer[i], "opacity": float(opacity[i])})
            view.addSphere({"center": {"x": x, "y": y, "z": z}, "radius": 0.22,
                            "color": inner[i]})

        for i, (sym, pos) in enumerate(zip(symbols, frame_pos)):
            view.addLabel(tracked_labels.get(i, sym), {
                "position": {"x": float(pos[0]), "y": float(pos[1]), "z": float(pos[2])},
                "fontSize": 12 if i in tracked_labels else 11,
                "fontColor": "black",
                "showBackground": False,
                "alignment": "center",
            })

        view.zoomTo()
        return view

    viewer_out, info_out, err_out = widgets.Output(), widgets.Output(), widgets.Output()
    cb_mag_out, cb_dir_out = widgets.Output(), widgets.Output()
    with cb_mag_out:
        display(colorbar_figure(mcolors.Normalize(0.0, float(mag_vmax)), CMAP_MAG,
                                "Inner — relative |F| error"))
    with cb_dir_out:
        display(colorbar_figure(mcolors.Normalize(-1.0, 1.0), CMAP_DIR_CB,
                                "Outer — cosine similarity"))

    # Wide + short, so the error trace sits as a strip under the viewer row.
    err_fig, err_ax = plt.subplots(figsize=(9.5, 2.0))
    err_ax.plot(rel_mag.mean(axis=1), color="darkmagenta")
    err_ax.set_xlim(0, n_frames - 1)
    err_ax.set_xlabel("frame", fontsize=9)
    err_ax.set_ylabel("|F| error", color="darkmagenta", fontsize=9)
    err_ax.tick_params(axis="y", labelcolor="darkmagenta", labelsize=8)
    err_ax.tick_params(axis="x", labelsize=8)
    err_ax2 = err_ax.twinx()
    err_ax2.plot(cos_sim.mean(axis=1), color="firebrick", alpha=0.7)
    err_ax2.set_ylabel("cos sim", color="firebrick", fontsize=9)
    err_ax2.tick_params(axis="y", labelcolor="firebrick", labelsize=8)
    err_ax2.set_ylim(-1.05, 1.05)
    err_marker = err_ax.axvline(0, color="black", linewidth=1.2)
    err_fig.tight_layout()
    plt.close(err_fig)

    def render_frame(frame_idx: int) -> None:
        view = build_view(frame_idx)
        viewer_out.clear_output(wait=True)
        with viewer_out:
            view.show()

        err_marker.set_xdata([frame_idx, frame_idx])
        err_out.clear_output(wait=True)
        with err_out:
            display(err_fig)

        mag, cos = rel_mag[frame_idx], cos_sim[frame_idx]
        d_energy = record["pred_energy"][frame_idx] - record["ref_energy"][frame_idx]  # type: ignore[index]
        info_out.clear_output(wait=True)
        with info_out:
            display(HTML(
                f"<div style='font-family:sans-serif;font-size:13px;line-height:1.7;padding-left:8px'>"
                f"<b>{record['formula']}<br>{split} · {Path(molecule).stem}</b><br>"
                f"model = SIMURGH ({metadata['model_class']})<br>"
                f"weights = {CKPT_DIR.parent.name}<br><br>"
                f"frame = {frame_idx}/{n_frames - 1}<br>"
                f"mean |F| err&nbsp;= {mag.mean():.4f}<br>"
                f"max&nbsp;|F| err&nbsp;= {mag.max():.4f}<br>"
                f"mean cos sim&nbsp;= {cos.mean():.4f}<br>"
                f"min&nbsp;cos sim&nbsp;= {cos.min():.4f}<br>"
                f"ΔE = {d_energy:+.4f} eV<br>"
                f"atoms = {n_atoms} · fragments = {max(record['fragment_index']) + 1}"  # type: ignore[operator]
                f"</div>"
            ))

    slider = widgets.IntSlider(min=0, max=n_frames - 1, step=1, value=0, description="frame")
    play = widgets.Play(min=0, max=n_frames - 1, step=1, interval=int(1000 / PLAYBACK_FPS))
    widgets.jslink((play, "value"), (slider, "value"))
    slider.observe(lambda change: render_frame(change["new"]), names="value")

    render_frame(0)
    display(widgets.HBox([play, slider]))
    display(widgets.HBox([viewer_out, cb_mag_out, cb_dir_out, info_out]))
    display(err_out)


print("viewer ready")

## 5 · The gallery

Five trajectories, five viewers. Press **▶** or drag the slider; each frame is re-rendered
on demand, so the first prediction pass (printed below each call) happens once and the
playback afterwards is pure drawing.

### In-distribution — monocarboxylic acids (same family as the training split)

#### O2C4H8 — shortest ID chain

In [ ]:
show_trajectory("TestID", "O2C4H8_PBE.traj")

#### O2C10H20 — longest ID chain

In [ ]:
show_trajectory("TestID", "O2C10H20_PBE.traj")

### Out-of-distribution — dicarboxylic acids (functional group duplicated)

#### O4C4H6 — shortest OOD chain

In [ ]:
show_trajectory("TestOOD", "O4C4H6_PBE.traj")

#### O4C7H12 — mid-chain OOD

In [ ]:
show_trajectory("TestOOD", "O4C7H12_PBE.traj")

#### O4C10H18 — longest OOD chain

In [ ]:
show_trajectory("TestOOD", "O4C10H18_PBE.traj")

## 6 · Summary across the five trajectories

Reuses the cached predictions — nothing is recomputed. ID rows are the accuracy the model
was validated at; OOD rows are the compositional-generalisation gap the benchmark exists
to measure.

In [ ]:
rows = [evaluate_trajectory(split, molecule) for split, molecule in TRAJECTORIES]

header = (f"{'split':8s} {'molecule':10s} {'atoms':>5s} {'frames':>6s} "
          f"{'E MAE/atom':>11s} {'F MAE':>9s} {'F max err':>9s} {'mean cos':>9s} {'min cos':>8s}")
print(header)
print("-" * len(header))
for r in rows:
    max_component = max(float(np.abs(p - q).max()) for p, q in zip(r["pred_forces"], r["ref_forces"]))
    print(f"{r['split']:8s} {r['formula']:10s} {r['n_atoms']:5d} {r['n_frames']:6d} "
          f"{r['energy_mae_per_atom']:11.5f} {r['forces_mae']:9.5f} {max_component:9.4f} "
          f"{np.mean(r['cos_sim']):9.5f} {np.min(r['cos_sim']):8.4f}")
print("\nunits: eV/atom, eV/Å, eV/Å")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for r in rows:
    style = "-" if r["split"] == "TestID" else "--"
    axes[0].plot(r["rel_mag"].mean(axis=1), style, linewidth=1.2, label=f"{r['split']} {r['formula']}")
    axes[1].plot(r["cos_sim"].mean(axis=1), style, linewidth=1.2)
axes[0].set_xlabel("frame")
axes[0].set_ylabel("mean relative |F| error")
axes[0].set_title("Force-magnitude error per frame")
axes[0].grid(alpha=0.3)
axes[1].set_xlabel("frame")
axes[1].set_ylabel("mean cos(F_pred, F_ref)")
axes[1].set_title("Force-direction agreement per frame")
axes[1].grid(alpha=0.3)
axes[0].legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.show()

## 7 · The same model as a plain ASE calculator

Everything above drives the module directly for speed. For ordinary ASE workflows
(`BFGS`, `Langevin`, NEB, phonons) you want a `Calculator`. `GOALCalculator` almost gets
there, but — as in section 2 — it builds graphs without `fragment_index`, which this
backbone requires. The subclass below reuses all of its loading machinery and overrides
only graph construction.

Worth upstreaming into `goal.ml.utils.calculator`: `GOALCalculator` could read
`data.fragment_*` from the checkpoint config it already loads, and then any
fragment-aware SIMURGH model would work with ASE out of the box.

In [ ]:
from ase.calculators.calculator import Calculator, all_changes
from ase.optimize import BFGS


class SimurghCalculator(GOALCalculator):
    """`GOALCalculator` that attaches the fragment labels the ARACE backbone needs.

    Parameters are identical to the parent, plus `fragment_kwargs` — pass the
    `FRAGMENT_KWARGS` derived from the checkpoint config so ASE-side graphs match the
    training-side ones.
    """

    def __init__(self, *args, fragment_kwargs: dict | None = None, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self._fragment_kwargs = dict(fragment_kwargs or {})

    def calculate(self, atoms=None, properties=None, system_changes=all_changes) -> None:
        Calculator.calculate(
            self, atoms, properties or self.implemented_properties, system_changes
        )
        graph = AtomicGraph.from_ase(
            self.atoms,
            cutoff=self._cutoff,
            dtype=self.dtype,
            head=self.head,
            **self._fragment_kwargs,
        ).to(self.device)
        with torch.enable_grad():
            predictions = self._module(graph)
        self.results["energy"] = predictions["energy"].detach().cpu().item()
        self.results["forces"] = predictions["forces"].detach().cpu().numpy()


# `fragment_on_failure="fallback"` matters here: an optimiser or thermostat walks through
# geometries rdkit cannot perceive (this is exactly where `raise` blows up mid-relaxation),
# and the library then falls back to connected-component labels with a one-time warning.
ase_calc = SimurghCalculator(
    checkpoint_path=CKPT_DIR,
    checkpoint="best",
    device=DEVICE,
    fragment_kwargs={**FRAGMENT_KWARGS, "fragment_on_failure": "fallback"},
)

# Relax the most out-of-distribution structure in the gallery.
relaxed = ase.io.read(str(DATA_ROOT / "TestOOD" / "O4C10H18_PBE.traj"), index=0)
relaxed.calc = ase_calc
e_start = relaxed.get_potential_energy()
f_start = np.abs(relaxed.get_forces()).max()

opt = BFGS(relaxed, logfile=None)
converged = opt.run(fmax=0.05, steps=25)

print(f"{relaxed.get_chemical_formula()} · SIMURGH-driven BFGS relaxation")
print(f"  start : E = {e_start:12.4f} eV   max|F| = {f_start:.4f} eV/Å")
print(f"  end   : E = {relaxed.get_potential_energy():12.4f} eV   "
      f"max|F| = {np.abs(relaxed.get_forces()).max():.4f} eV/Å")
print(f"  ΔE = {relaxed.get_potential_energy() - e_start:+.4f} eV over {opt.nsteps} steps "
      f"(converged: {converged})")

---

### What to take away

| | |
|---|---|
| Model | SIMURGH-ARACE, 2 rounds, `16x0e + 16x1o + 16x2e` artisans, fragment interaction on |
| Checkpoint | `logs/simurgh_gmd26/runs/2026-08-03_16-52-12_…/checkpoints`, best `val_forces_mae` |
| Loading | `GOALCalculator(checkpoint_path=<dir>, checkpoint="best")` → frozen-source rebuild |
| Graphs | `AtomicGraph.from_ase(..., compute_fragment_index=True, fragment_*=…)` — required by this backbone |
| Data | `data/GMD/FragmentDuplication/{TestID,TestOOD}`, 200 frames per trajectory |

`PREDICTIONS` holds every array computed above (`ref_forces`, `pred_forces`, `rel_mag`,
`cos_sim`, energies) keyed by `(split, molecule)` — add a cell and dig in without
re-running the model. To point the gallery at a different run, change `CKPT_DIR` in
section 0 and restart; to add or swap molecules, edit `TRAJECTORIES`.